# BiblioIA

Es un sistema de gestión de biblioteca
que combina una base de datos relacional robusta con un agente de inteligencia artificial capaz
de:
- Comprender preguntas formuladas en lenguaje natural (español).
- Traducirlas a consultas SQL válidas sobre el esquema de la biblioteca.
- Ejecutar esas consultas y devolver resultados interpretables.
- Generar recomendaciones de libros personalizadas basadas en el historial del usuario.

## Instalar dependencias

Primero es necesario instalar todas las dependencias para usar el agente, para ello es necesario ejecutar la siguiente celda

In [ ]:
!pip install --user langchain langchain-community langchain-groq pymysql pandas

## Conectar la base de datos

Para realizar la conexion es necesario generar un archivo .env con los siguientes campos para una base de datos de mysql

- GROQ_API_KEY donde se guardara la apikey de groq
- DB_USER usuario de la base de datos
- DB_PASSWORD contraseña del usuario
- DB_HOST localhost si es local
- DB_PORT puerto de la base de datos (por defecto es  3306)
- DB_NAME nombre de la base de datos

__NOTA: La creacion de la base de datos debe ser manual__

Luego se debera correr la siguiente celda para verificar si la conexion fue exitosa.


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase

# 1. Cargar automáticamente el archivo .env
load_dotenv()

# 1. Configuración de tu API Key de Groq (Consíguela gratis en console.groq.com)
#if "GROQ_API_KEY" not in os.environ:
#    os.environ["GROQ_API_KEY"] = getpass("Introduce tu Groq API Key: ")

# 2. Configura las credenciales de tu base de datos MySQL local o remota
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# Crear el URI de conexión con el dialecto pymysql
db_uri = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

# Instanciar el conector relacional de LangChain
db = SQLDatabase.from_uri(db_uri)

# Verificación opcional: Imprime las tablas legibles encontradas
print("✅ Conectado a MySQL. Tablas detectadas:", db.get_usable_table_names())

## Configuracion del agente

Ahora se debe ejecutar la siguiente celda para que el agente configure las consultas text-to-sql de acuerdo a las preguntas solicitadas y devuelva una respuesta limpia

In [ ]:
from operator import itemgetter
from langchain_groq import ChatGroq
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
# 🌟 IMPORTACIÓN NUEVA: Agregamos Markdown y clear_output
from IPython.display import display, HTML, Markdown, clear_output

# 1. Inicializar el LLM de Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# 2. Herramienta para interactuar y ejecutar código en MySQL
execute_query_tool = QuerySQLDatabaseTool(db=db)

# 3. Prompt técnico estricto
sql_generation_prompt = PromptTemplate.from_template(
    """Dada una base de datos MySQL con el siguiente esquema de tablas, escribe una consulta SQL válida que responda a la pregunta del usuario. 
Devuelve exclusivamente la consulta SQL limpia, sin bloques de código en Markdown (como ```sql), sin explicaciones adicionales y sin texto antes o después.

Esquema de la Base de Datos:
{schema}

Pregunta del usuario: {question}
Consulta SQL MySQL:"""
)

def limpiar_sql(texto_sql: str) -> str:
    limpio = texto_sql.replace("```sql", "").replace("```", "")
    return limpio.strip()

# 4. Cadena de generación
generate_query_chain = (
    RunnablePassthrough.assign(schema=lambda _: db.get_table_info())
    | sql_generation_prompt
    | llm
    | StrOutputParser()
    | limpiar_sql 
)

# 5. 🌟 ACTUALIZADO: Prompt de formateo final obligando a usar tablas
answer_prompt = PromptTemplate.from_template(
    """Dado el contexto de la base de datos MySQL, la pregunta del usuario, la consulta SQL ejecutada y el resultado crudo obtenido, redacta una respuesta clara, concisa y profesional en español basándote en los datos reales arrojados.

⚠️ REGLA ESTRICTA: Si el resultado de la base de datos contiene una lista de elementos, múltiples registros o datos comparativos, debes estructurar la respuesta utilizando obligatoriamente una tabla de Markdown.

Pregunta original: {question}
Consulta SQL generada: {query}
Resultado de la base de datos: {result}

Respuesta final en español:"""
)

# 6. Pipeline Completo
full_chain = (
    RunnablePassthrough.assign(query=generate_query_chain)
    .assign(result=itemgetter("query") | execute_query_tool)
    | answer_prompt
    | llm
    | StrOutputParser()
)

# 7. 🌟 ACTUALIZADO: Función contenedora procesando Markdown
def preguntar(pregunta_usuario: str):
    # Limpia la pantalla de la celda antes de mostrar el nuevo resultado
    clear_output(wait=True) 
    display(HTML(f"<b>👤 Pregunta:</b> <i>{pregunta_usuario}</i><br>⏳ <i>Consultando la base de datos...</i>"))
    
    try:
        respuesta = full_chain.invoke({"question": pregunta_usuario})
        display(HTML("<hr><b>🤖 Respuesta del Asistente:</b><br>"))
        
        # Procesamos la respuesta del LLM a través de Markdown en lugar de HTML
        display(Markdown(respuesta))
        
    except Exception as e:
        display(HTML(f"<hr><b style='color:red;'>❌ Error:</b> {e}"))

display(HTML("<div style='background-color: #d4edda; color: #155724; padding: 10px; border-radius: 5px; border: 1px solid #c3e6cb;'>✅ <b>¡Motor Text-To-SQL cargado exitosamente!</b>.</div>"))

## DEMO

Puede realizar la siguiente consulta para verificar si el agente funciona

In [ ]:
preguntar("¿Cuantos libros hay disponibles?")